# Advisor Match Agent: the three-stage workflow

This notebook explains and exercises the repository's **current implementation**: flexible upload interpretation, deterministic advisor matching, conversational exception review, and verified workbook generation.

> It intentionally makes no model or external database call. Executable examples use the checked-in synthetic source and the same deterministic Python modules as the application. Candidate ordering is explicit and deterministic; the runtime model and workbook never receive internal firm-similarity values.

The central boundary is simple: **the model interprets bounded evidence and conducts the conversation; deterministic application code owns identity decisions and workbook mutations.**

## Learning goals and setup

By the end, you should be able to explain:

1. how headed, later-header, and headerless uploads become an exact `InputMapping`;
2. why mapping validation and the missing-firm checkpoint happen before reference retrieval;
3. how one opaque authoritative snapshot is persisted and reused for an immutable attachment;
4. how CRD, email, name, firm, city, state, ZIP, nicknames, and conflicts affect deterministic decisions;
5. how bounded review, explicit overrides, audit history, and workbook regeneration work.

From the repository root, run `uv sync --locked --all-groups`, open this notebook, and select that environment.

In [ ]:
from __future__ import annotations

import ast
import inspect
import sys
import tempfile
import textwrap
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
import yaml
from openpyxl import load_workbook


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / 'general_agent' / 'agent.py').is_file():
            return path
    raise RuntimeError('Start Jupyter in the advisor-match-agent repository.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## 1. Architecture and responsibility boundary

```mermaid
flowchart TD
    U["One uploaded CSV or XLSX"] --> P["Bounded raw profiler"]
    P --> A{"One clear interpretation?"}
    A -->|"no"| Q["Ask the user"]
    Q --> A
    A -->|"yes"| V["Validate exact sheet, header row, indexes, and headers"]
    V --> R["create_advisor_match + resolve firm intent"]
    R --> F{"Firm discrepancy or weak name-only rows?"}
    F -->|"yes"| C["Ask for source, override, or explicit continue"]
    C --> R
    F -->|"no"| S
    S[("Protected opaque reference snapshot")]
    S --> M
    M --> D[("Corporation-scoped session and audit")]
    M --> W["Verified advisor_matches.xlsx"]
    D --> E["Bounded exception pages"]
    E --> X{"Explicit user decision?"}
    X -->|"yes"| D
    D --> W
```

The model never sees the complete upload, authoritative table, persisted decision collection, protected snapshot path, or workbook internals. Application code enforces corporation/conversation scope, path and hash integrity, row limits, policy, audit, and export verification.

### The explicit graph and two bounded LLM decisions

`general_agent/graph.py` compiles one `StateGraph`. The model is used only for a strict request route and a strict upload mapping. There is no planner, tool loop, filesystem skill, or subagent; deterministic modules remain independently testable.

In [ ]:
from general_agent.graph_state import MappingDecision, RouteDecision

structured_contracts = pd.DataFrame([
    {'decision': 'route', 'schema': RouteDecision.__name__, 'attempts': 3},
    {'decision': 'mapping', 'schema': MappingDecision.__name__, 'attempts': 3},
])
structured_contracts

The graph is deliberately ordered around explicit stages:

1. `route`, `inspect`, `map_input`, and deterministic `validate`;
2. deterministic `match`, which resolves firm intent, retrieves or reuses the attachment snapshot, scores, persists, and publishes;
3. verified workbook publication, followed only by `reset`, `capabilities`, and `unsupported` terminal guidance branches.

Mapping or firm ambiguity uses `interrupt()` and the next text-only turn resumes with `Command(resume=...)`. A new immutable attachment resets the in-memory thread and triggers one complete source iteration.

## 2. Stage 1 — interpret the upload flexibly

The profiler reads raw physical rows with `header=None`. For each bounded sheet it returns physical row numbers, up to five plausible header candidates, exact column indexes and observed headers, small post-header samples, patterns, and a headerless view. Suggestions are clues—not authority. If multiple sheets, header rows, or meanings remain plausible, the agent asks the user.

A headed mapping uses a one-based `header_row` and exact zero-based `(index, header)` references. A headerless mapping uses `header_row=None` and `header=None`. The validator rechecks those references before loading any mapped rows and returns a fingerprint over the source bytes plus canonical mapping.

In [ ]:
from general_agent.advisor_matching.input_loader import validate_and_load_input
from general_agent.advisor_matching.profiler import inspect_advisor_upload
from general_agent.advisor_matching.schemas import InputMapping
from general_agent.config import Settings

tutorial_settings = Settings(project_root=PROJECT_ROOT, model_name='tutorial:no-model')
later_header_path = PROJECT_ROOT / 'examples' / 'advisor-match' / 'preamble_and_header.xlsx'
later_profile = inspect_advisor_upload(later_header_path, tutorial_settings)
later_sheet = later_profile['sheets'][0]
pd.DataFrame(later_sheet['preview_rows'])

In [ ]:
later_mapping = InputMapping.model_validate({
    'sheet_name': 'Advisors',
    'header_row': 3,
    'full_name': {'columns': [{'index': 0, 'header': 'Advisor Name'}]},
    'firm_name': {'columns': [{'index': 1, 'header': 'Organization'}]},
    'city': {'columns': [{'index': 2, 'header': 'Town'}]},
    'state': {'columns': [{'index': 3, 'header': 'Province'}]},
})
loaded = validate_and_load_input(later_header_path, later_mapping, max_rows=50_000)
assert [row[0] for row in loaded.rows] == [4, 5]
print('Validated columns:', loaded.columns)
print('Mapping fingerprint:', loaded.mapping_fingerprint)
print('Input summary:', loaded.summary.model_dump())
pd.DataFrame([{'physical_row': number, **mapped} for number, _, mapped in loaded.rows])

### Headerless input and the missing-firm checkpoint

Generated labels such as `Column A` are preview/display labels only. Exact headerless bindings still contain `header=None`. Completely blank rows are skipped while physical source row numbers are retained. Preamble rows above a selected header are not data.

After validation, any row with a usable multi-token name but no normalized firm, valid CRD, or valid email triggers one conversational checkpoint. A same-turn all-rows firm is applied to copied mapped values inside `create_advisor_match`; no upload is changed or derived. The user may instead explicitly continue with weaker evidence.

In [ ]:
headerless_path = PROJECT_ROOT / 'examples' / 'advisor-match' / 'headerless_advisors.csv'
headerless_mapping = InputMapping.model_validate({
    'header_row': None,
    'crd_number': {'columns': [{'index': 0, 'header': None}]},
    'first_name': {'columns': [{'index': 1, 'header': None}]},
    'last_name': {'columns': [{'index': 2, 'header': None}]},
    'firm_name': {'columns': [{'index': 3, 'header': None}]},
    'city': {'columns': [{'index': 4, 'header': None}]},
    'state': {'columns': [{'index': 5, 'header': None}]},
})
headerless_loaded = validate_and_load_input(headerless_path, headerless_mapping, max_rows=50_000)
assert headerless_loaded.columns[0]['header'] is None
print(headerless_loaded.columns)

with tempfile.TemporaryDirectory(prefix='advisor-match-preflight-') as directory:
    name_only_path = Path(directory) / 'name-only.csv'
    name_only_path.write_text('Name\nRobert Mercer\n', encoding='utf-8')
    name_only_mapping = InputMapping.model_validate({
        'full_name': {'columns': [{'index': 0, 'header': 'Name'}]},
    })
    preflight = validate_and_load_input(name_only_path, name_only_mapping, max_rows=100)
assert preflight.summary.missing_firm_confirmation_required is True
preflight.summary.model_dump()

## 3. Stage 2 — match deterministically

Only after mapping clarification does the agent call `create_advisor_match`. During the first call for an immutable attachment, the workflow projects and validates the complete authoritative source, atomically stores its protected corporation-and-conversation-scoped snapshot, and builds compact exact CRD, email, and first/last-name indexes in the same stream. Retries and mapping corrections rebuild the temporary index from that local snapshot without querying the authoritative source again.

`create_advisor_match` returns the opaque snapshot manifest with the session result, runs policy version 5, persists structured decisions, and generates the workbook. The compact index is released after decisions are persisted. Malformed individual values become warnings or row-level `No Match` reasons. Only structural input, mapping, limit, or reference-integrity problems stop the run.

In [ ]:
from general_agent.advisor_matching.schemas import MASTER_COLUMNS
from general_agent.advisor_matching.source import SyntheticAdvisorReferenceSource

master_path = PROJECT_ROOT / 'general_agent' / 'advisor_matching' / 'data' / 'master_advisors.csv'
advisors = list(SyntheticAdvisorReferenceSource(master_path).iter_records())
assert 'STREET_ADDRESS' not in MASTER_COLUMNS
print('Projected authoritative columns:', MASTER_COLUMNS)
print('Validated synthetic advisors:', len(advisors))
pd.DataFrame([advisor.model_dump() for advisor in advisors]).head(6)

### Decision order and evidence gates

1. Exact CRD is decisive; conflicting values become warnings.
2. A unique normalized email is strong. A non-unique authoritative email is `Ambiguous Match`.
3. A usable name is a multi-token full name or both first and last name.
4. Exact normalized first/last candidates are retrieved from the name index; middle tokens in full names are ignored. No full-master fuzzy-name scan is performed.
5. An exact name still needs independent exact/wildcard/close firm or exact city+state support. Firm similarity is evaluated only inside that bounded name group.
6. Strong firm or state conflicts block name-based automation. A city difference within the same state is weaker.
7. Curated nickname and name-only evidence may generate candidates but never auto-match.
8. ZIP is display-only context: it contributes no support or conflict. Street address is outside the matching domain.
9. A name typo with no exact indexed candidate becomes `No Match / NAME_NOT_FOUND`. Unresolved candidates become `Ambiguous Match`.

Candidates use explicit precedence: raw exact name before nickname, conflict-free before conflicted, then exact firm, exact city/state, wildcard firm, close firm, and CRD. At most three are exposed, with the full pool size and truncation flag retained.

In [ ]:
from general_agent.advisor_matching import normalization as norm
from general_agent.advisor_matching.policy import (
    FIRM_CONFLICT_SIMILARITY, MINIMUM_FIRM_SIMILARITY, POLICY_VERSION,
    FIRM_WILDCARD_MIN_LENGTH, REVIEW_CANDIDATE_LIMIT,
)

normalization_examples = pd.DataFrame([
    {'field': 'crd', 'raw': ' 99000006.0 ', 'normalized': norm.crd(' 99000006.0 ')},
    {'field': 'email', 'raw': ' Robert.Mercer@Example.COM ', 'normalized': norm.email(' Robert.Mercer@Example.COM ')},
    {'field': 'person_name', 'raw': 'Dr. Róbert Mercer, Jr.', 'normalized': norm.person_name('Dr. Róbert Mercer, Jr.')},
    {'field': 'firm', 'raw': 'Morgan Stanley & Co., LLC', 'normalized': norm.firm('Morgan Stanley & Co., LLC')},
    {'field': 'state', 'raw': 'Massachusetts', 'normalized': norm.state('Massachusetts')},
    {'field': 'zip context', 'raw': '02108-1200', 'normalized': norm.zip_code('02108-1200')},
])
policy = {
    'version': POLICY_VERSION,
    'minimum_firm_similarity': MINIMUM_FIRM_SIMILARITY,
    'firm_conflict_similarity': FIRM_CONFLICT_SIMILARITY,
    'wildcard_minimum_length': FIRM_WILDCARD_MIN_LENGTH,
    'candidate_limit': REVIEW_CANDIDATE_LIMIT,
}
documented_policy = yaml.safe_load((PROJECT_ROOT / 'docs' / 'contracts' / 'matching-policy.yaml').read_text())
assert documented_policy['version'] == POLICY_VERSION
assert documented_policy['name_matching']['fuzzy_names'] is False
print(policy)
normalization_examples

### Worked decisions against the real synthetic source

The next cell calls the same `run_matching` function used by the workflow tool. Qualitative candidate evidence, total candidate count, and truncation state are visible; no internal candidate-score fields exist.

In [ ]:
from general_agent.advisor_matching.matcher import run_matching

MAPPED_FIELDS = (
    'crd_number', 'first_name', 'last_name', 'full_name', 'firm_name',
    'email', 'city', 'state', 'zip_code',
)


def tutorial_row(row_number: int = 2, **values: str):
    mapped = {field: '' for field in MAPPED_FIELDS}
    mapped.update(values)
    return row_number, dict(mapped), mapped


cases = {
    'exact CRD wins despite conflicts': {'crd_number': '99000006', 'full_name': 'Someone Else', 'email': 'other@example.com', 'state': 'CA'},
    'unique email with unknown CRD': {'crd_number': '99999999', 'email': ' ELIZABETH.HART@EXAMPLE.COM '},
    'exact name plus legal-suffix firm': {'full_name': 'Avery Stone', 'firm_name': 'Northstar Wealth Partners, LLC'},
    'name typo is not fuzzy matched': {'full_name': 'John Smyth', 'firm_name': 'Summit Ridge Advisors', 'city': 'Denver', 'state': 'Colorado'},
    'nickname plus support is review-only': {'full_name': 'Bob Mercer', 'firm_name': 'Cedar Grove Advisory', 'city': 'Richmond', 'state': 'VA'},
    'name-only is review-only': {'full_name': 'John Smith'},
    'ZIP cannot support a name': {'full_name': 'John Smith', 'zip_code': '02108'},
    'malformed CRD only': {'crd_number': '99-000-006'},
    'firm/location without identity': {'firm_name': 'Cedar Grove Advisory', 'city': 'Richmond', 'state': 'VA'},
    'unknown named advisor': {'full_name': 'Quinn Example', 'firm_name': 'Imaginary Finance', 'city': 'Albany', 'state': 'NY'},
}

summary_rows = []
for label, values in cases.items():
    decision = run_matching([tutorial_row(**values)], advisors)[0][0]
    summary_rows.append({
        'case': label, 'status': decision.status, 'rule': decision.rule_id,
        'matched_crd': decision.matched_advisor.crd_number if decision.matched_advisor else None,
        'candidate_crds': ', '.join(candidate.crd_number for candidate in decision.candidates),
        'warnings': '; '.join(decision.warnings),
    })
case_summary = pd.DataFrame(summary_rows)
assert case_summary.loc[0, 'rule'] == 'EXACT_CRD'
assert case_summary.loc[4, 'status'] == 'Ambiguous Match'
assert case_summary.loc[7, 'rule'] == 'MALFORMED_CRD'
case_summary

### Duplicate rows and row-level outcomes

Duplicate signatures use normalized CRD, email, name, firm, city, state, and ZIP. Duplicates remain separate decisions with distinct review item IDs and physical source rows; a shared duplicate-group marker is an audit flag, not a deduplication action.

In [ ]:
duplicate_decisions, duplicate_counts, duplicate_warnings = run_matching([
    tutorial_row(2, crd_number='99000006'),
    tutorial_row(3, crd_number='99000006'),
], advisors)
assert duplicate_decisions[0].review_item_id != duplicate_decisions[1].review_item_id
assert duplicate_decisions[0].duplicate_group == duplicate_decisions[1].duplicate_group
print(duplicate_counts.model_dump(), duplicate_warnings)
pd.DataFrame([{
    'source_row': item.source_row_number, 'review_item_id': item.review_item_id,
    'duplicate_group': item.duplicate_group, 'status': item.status,
} for item in duplicate_decisions])

## 4. Stage 3 — hand off exceptions in the workbook

After matching, the application reports the three counts and publishes one verified workbook. `Review Required` contains physical source rows, candidate CRDs, and qualitative supporting/conflicting/context evidence—never numeric scores.

The graph ends after publication. The reviewer can record an outcome in the blank `User Decision`, `Selected CRD`, and `Reviewer Notes` columns in a downloaded copy. Those edits are not re-ingested or validated by the application. Source-value corrections require a new upload and matching run.

In [ ]:
offline_review_columns = ['User Decision', 'Selected CRD', 'Reviewer Notes']
pd.DataFrame({'editable_review_column': offline_review_columns})

## 5. Human-first, auditable workbook

Each matching run generates exactly four sheets: `Matched`, `Review Required`, `Original Input`, and `Run Summary`. `Matched` exposes 17 human-facing columns; `Review Required` exposes 23, including three highlighted offline-review fields, candidate pool size, and whether the visible three-candidate list was truncated. Technical IDs, rules, confidence, and duplicate group remain hidden at the right. Names and locations are combined for readability.

The generator adds filters, frozen headers, wrapping, alternating fills, status colors, bounded column widths at least as wide as headers, and capped result-row heights. CRD/ZIP and every user-controlled string are forced to text to prevent formula injection. Verification reopens the file, checks sheet order, rejects formulas, reconciles all decisions to Original Input, and validates all three persisted status counts before atomic publication.

In [ ]:
from general_agent.advisor_matching.matcher import run_matching
from general_agent.advisor_matching.schemas import ReferenceSnapshotManifest
from general_agent.advisor_matching.source import sha256_file
from general_agent.advisor_matching.workbook import verify_match_workbook, write_match_workbook

fixture_decisions, fixture_counts, fixture_warnings = run_matching(loaded.rows, advisors)
reference_manifest = ReferenceSnapshotManifest(
    reference_snapshot_id='ars_' + 'a' * 32,
    row_count=len(advisors), columns=list(MASTER_COLUMNS),
    source_kind='synthetic', schema_version='1',
    retrieved_at=datetime.now(UTC), sha256=sha256_file(master_path),
)

with tempfile.TemporaryDirectory(prefix='advisor-match-tutorial-') as directory:
    workbook_path = Path(directory) / 'advisor_matches.xlsx'
    write_match_workbook(
        workbook_path, session_id='ams_tutorial', decisions=fixture_decisions,
        counts=fixture_counts, mapping=later_mapping, input_summary=loaded.summary,
        source_name=later_header_path.name, source_sha256=loaded.source_sha256,
        reference=reference_manifest, policy_version=POLICY_VERSION,
    )
    verification = verify_match_workbook(
        workbook_path, expected_rows=len(loaded.rows), expected_counts=fixture_counts,
    )
    workbook = load_workbook(workbook_path, data_only=False)
    sheet_names = tuple(workbook.sheetnames)
    matched_headers = [cell.value for cell in workbook['Matched'][1]]
    visible_matched = [
        header for index, header in enumerate(matched_headers, start=1)
        if not workbook['Matched'].column_dimensions[workbook['Matched'].cell(1, index).column_letter].hidden
    ]
    formula_cells = [
        cell.coordinate for sheet in workbook.worksheets for row in sheet.iter_rows()
        for cell in row if cell.data_type == 'f'
    ]
    workbook.close()
assert len(visible_matched) == 17
assert not formula_cells
print('Counts:', fixture_counts.model_dump(), 'Warnings:', fixture_warnings)
print('Sheets:', sheet_names, 'Verification:', verification)

## 6. Concrete graph trace

| Step | Actor | Action | Model-visible result |
|---:|---|---|---|
| 1 | Router node | Classify current request and extract explicit firm instructions | Typed route only |
| 2 | Inspect node | Inspect one upload | Bounded raw/header/headerless evidence |
| 3 | Mapping node | Choose a clear interpretation or interrupt | Typed mapping or one question |
| 4 | Validate node | Bind exact sheet/index/header | Mapping, fingerprint, summary, missing-firm sample |
| 5 | Match node | Resolve firm context and run matching | Match, bounded clarification, or source blocker |
| 6 | User if needed | Choose source, restated override, or weaker-evidence continuation | Explicit direction |
| 7 | Application | Retrieve once or reuse attachment snapshot; build compact index | No model participation |
| 8 | Application | Match, persist, generate, verify, and publish | No model participation |
| 9 | User | Download and review `Review Required` locally | Outside application boundary |

## 7. Current boundaries and source map

The source adapter currently uses a 40-row synthetic CSV. Production Snowflake should implement the same projected streaming schema and opaque manifest: one complete projected query per new immutable attachment, then local snapshot reuse. The application already avoids the uploaded-rows × master-rows scan by building compact integer-posting indexes for exact CRD, email, and normalized first/last name. Corporation IDs isolate storage but are not authentication, so both services remain loopback-only. Advisor profile reporting is supported only as deterministic blank placeholder HTML from validated CRDs; no profile data is fetched or simulated.

Key files:

- `general_agent/graph.py` — explicit nodes, edges, routing, and interrupts;
- `general_agent/graph_state.py` — graph state and strict LLM output contracts;
- `general_agent/advisor_service.py` — explicit upload, snapshot, matching, review, and workbook service;
- `general_agent/advisor_matching/schemas.py` — mapping, snapshot, decision, and review contracts;
- `profiler.py` / `input_loader.py` — bounded interpretation and exact validation;
- `normalization.py` / `index.py` / `policy.py` / `matcher.py` — compact lookup and deterministic identity policy;
- `general_agent/advisor_repository.py` — durable snapshots, sessions, audits, proposals, and artifacts;
- `general_agent/advisor_matching/workbook.py` — four-sheet projection and verification;
- `docs/contracts/` — static policy, mapping, review, and workbook contracts;
- `tests/test_advisor_matching.py`, `test_advisor_service.py`, and `test_advisor_workbook.py` — executable edge cases.

A useful next exercise is to add a labeled edge case, predict its decisive rule, support/conflict gates, candidate set, and row-level reason, then encode that prediction as a deterministic test before adjusting the bounded firm-similarity threshold.